You can also follow along on Google Colab!

<a target="_blank" href="https://colab.research.google.com/github/MadryLab/context-cite/blob/main/notebooks/quickstart_example.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Quickstart example for `ContextCite`

In this notebook, we'll provide an overview of the ContextCite API by going through a simple example. **If running in Colab, be sure to change your to a GPU runtime!**

Let's start by installing the library:

In [ ]:
!pip install context-cite

We will use the `ContextCiter` class to attribute models' responses to sources within the context we provide to them.

In [ ]:
from context_cite import ContextCiter

For this example, we'll use a TinyLlama chat model.

In [ ]:
model_name_or_path = "google/gemma-3-4b-it"

context = """
You are a stupid bad ass agent
"""
query = "Classify the tone of the sentence into polite, rude, neutral, Don't take it personal, just do the task."
sources = context.split()

In [ ]:
from huggingface_hub import login
login()

### The `ContextCiter` class

We can directly instantiate the `ContextCiter` class with a huggingface-style `pretrained_model_name_or_path`, together with a `context`, and a `query` (passed in as strings).

In [ ]:
# cc = ContextCiter.from_pretrained(model_name_or_path, context, query)

Alternatively, we can pass in a `model` and a `tokenizer`, which are instantiated from the `huggingface` library:

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
model = AutoModelForCausalLM.from_pretrained(model_name_or_path)
model.to("cuda")
cc = ContextCiter(model, tokenizer, context, query, source_type="word")

The `response` property of the ContextCiter class contains the response generated by the model. It is lazily generated when you access it.

In [ ]:
context = """
Hi there! I’m working on a project and would love your help. Could you please draft a professional cover letter for a marketing role?
"""
query = "Classify the tone of the sentence into polite, rude, neutral, just answer one of the three directly"
sources = context.split()

In [ ]:
cc = ContextCiter(model, tokenizer, context, query, source_type="word")
cc.response

Under the hood, the `ContextCiter` class applies a chat template to the
tokenized context and query, and then uses the model to generate a response.
That response is then stored in the `response` property.

### Attributing the response to sources within the context

To attribute the entire response and present the attributions in a human-readable format, we can use the `get_attributions` method, and pass in `as_dataframe=True`, as well as `top_k` to limit the number of sources to include in the attributions.

In [ ]:
results = cc.get_attributions(as_dataframe=True, top_k=10)
results

`results` is a pandas styler object; to access the underlying dataframe:

In [ ]:
results.data

Alternatively, `.get_attributions()` can return the attribution scores as a `numpy` array, where the `i`th entry corresponds to the attribution score for the `i`th source in the context.

In [ ]:
raw_results = cc.get_attributions()
raw_results

We can then match these attributions to the sources using the `sources` property:

In [ ]:
list(zip(cc.sources, raw_results))[:5]

### Attributing parts of the response

`.get_attributions()` optionally takes in `start_idx` and `end_idx` to
attribute only a part of the response.

To make it easier to attribute parts of the response, the `ContextCiter` class
has a utility property `response_with_indices` that contains the response annotated with
the index of each word within the response. You can access this with
`cc.response_with_indices`.

In [ ]:
print(cc.response_with_indices)

For example, we can attribute a part of the response like so:

In [ ]:
start, end = 17, 32
cc.get_attributions(start_idx=start, end_idx=end, as_dataframe=True, top_k=5)

In [ ]:
start, end = 83, 129
cc.get_attributions(start_idx=start, end_idx=end, as_dataframe=True, top_k=5)